# IMPORT LIBRARY

In [ ]:
import datetime
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine
import psycopg2

import requests
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

import pymysql
pd.set_option('display.max_columns', None)

# CONFIG

In [ ]:
load_data_by = 'lahiardhan@jiwagroup.com'

current_hour_wib = datetime.now(timezone('Asia/Jakarta')).hour
if current_hour_wib < 11:
    batch_label = '08AM'
elif current_hour_wib < 15:
    batch_label = '11AM'
else:
    batch_label = '03PM'

print(f"Current WIB hour: {current_hour_wib} -> batch_label: {batch_label}")

In [ ]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dl_dwh = module_from_file("config_dw", "../db_config/config_db_datawarehouse.py")
config_jiwaplus = module_from_file("config_dw", "../db_config/config_db_jiwaplus.py")
config_ksj_link = module_from_file("config_dw", "../db_config/config_db_ksj_link.py")
config_ksj_link_location = module_from_file("config_dw", "../db_config/config_db_ksj_location.py")

In [ ]:
def jiwa_db_conn():
    return psycopg2.connect(
        host=config_jiwaplus.hostname,
        database=config_jiwaplus.database,
        user=config_jiwaplus.username,
        password=config_jiwaplus.password,
        port='5432'
    )

In [ ]:
def ksj_link_db_conn(db_name):
    return psycopg2.connect(
        host=config_ksj_link.hostname,
        database=db_name,
        user=config_ksj_link.username,
        password=config_ksj_link.password,
        port='5432'
    )

In [ ]:
def ksj_link_location_db_conn(db_name):
    return psycopg2.connect(
        host=config_ksj_link_location.hostname,
        database=db_name,
        user=config_ksj_link_location.username,
        password=config_ksj_link_location.password,
        port='5432'
    )

# FUNCTION

In [ ]:
def get_rider_total_working_time():
    connection = ksj_link_db_conn(config_ksj_link.d_jiwa_ksj)
    cursor = connection.cursor()
    
    query_1 = f"""
        SELECT 
            TO_CHAR(date, 'YYYY-MM-DD') as operating_date,
            rider_transfer_log.rider_code,
            checkin + INTERVAL '7 hours' as checkin,
            checkout + INTERVAL '7 hours' as checkout,
            reverse_request_at + INTERVAL '7 hours' as reversed_at,
            EXTRACT(EPOCH FROM (reverse_request_at - checkin)) / 60.0 as total_working_minutes,
            EXTRACT(EPOCH FROM (reverse_request_at - checkin)) / 60.0 / 60.0 as total_working_hour,
            EXTRACT(EPOCH FROM (checkout - reverse_request_at)) / 60.0 as total_reverse_process_admin
        FROM 
            rider_transfer_log
            JOIN rider_reverse_request ON rider_transfer_log.id = rider_reverse_request.rider_transfer_log_id 
        WHERE 
            (rider_transfer_log.checkin + INTERVAL '7 hours')::date = (NOW() + INTERVAL '7 hours')::date
            AND checkin IS NOT NULL
            AND rider_reverse_request.status = 'APPROVED'
    """
    
    cursor.execute(query_1)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()
    
    return df

In [ ]:
def get_rider_stationed_logs():
    connection = ksj_link_location_db_conn(config_ksj_link_location.d_location)
    cursor = connection.cursor()
    
    query_1 = f"""
        SELECT
            TO_CHAR(created_at, 'YYYY-MM-DD') as date,
            id,
            rider_code,
            latitude,
            longitude,
            is_stationed,
            created_at  + INTERVAL '7 hours' as timestamp
        FROM
            rider_stationed_logs
        WHERE
            (created_at + INTERVAL '7 hours')::date = (NOW() + INTERVAL '7 hours')::date
        ORDER BY created_at ASC
    """
    
    cursor.execute(query_1)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()
    
    return df

In [ ]:
def get_raw_transactions():
    connection = ksj_link_db_conn(config_ksj_link.d_transaction)
    cursor = connection.cursor()
    
    query_1 = f"""
        SELECT 
            id,
            transaction_datetime_utc,
            transaction_datetime_utc + INTERVAL '7 hour' AS transaction_datetime_wib,
            (transaction_datetime_utc + INTERVAL '7 hour')::date AS transaction_date,
            rider_code,
            rider_name,
            hub_code,
            total_item_qty,
            total_price_after_discount,
            payment_type
        FROM transactions
        WHERE 
            transaction_status = 'COMPLETED'
            AND (transaction_datetime_utc + INTERVAL '7 hours')::date = (NOW() + INTERVAL '7 hours')::date
    """
    
    cursor.execute(query_1)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()
    
    return df

In [ ]:
def get_hub_mapping():
    connection = ksj_link_db_conn(config_ksj_link.d_transaction)
    cursor = connection.cursor()
    
    query_1 = f"""
        SELECT DISTINCT
            d1.jilid,
            d1.hub_id AS hub_code,
            d1.hub_name,
            d2.business_unit,
            d2.region,
            d2.kota,
            d2.provinsi
        FROM d_sejutajiwa_riders_hubs_mapping d1
        JOIN d_outlet_mapping_custom d2 ON d1.jilid = d2.jilid
    """
    
    cursor.execute(query_1)

    record = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()
    
    return df

In [ ]:
def create_mangkal_session():
    # =========================
    # 1. EXTRACT & TRANSFORM
    # =========================
    df = get_rider_stationed_logs()

    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.sort_values(['rider_code', 'timestamp', 'id']).reset_index(drop=True)

    # Make sure is_stationed is boolean
    if df['is_stationed'].dtype != bool:
        df['is_stationed'] = df['is_stationed'].astype(str).str.lower().map({
            'true': True,
            'false': False
        })

    # Group by rider
    grp = df.groupby('rider_code', sort=False)

    # Start of session:
    # True that appears after previous row is not True
    prev_is_stationed = grp['is_stationed'].shift(fill_value=False)
    df['is_session_start'] = df['is_stationed'] & (~prev_is_stationed)

    # Session sequence per rider
    df['session_seq'] = df['is_session_start'].groupby(df['rider_code']).cumsum()

    # Create session_id
    df['session_id'] = np.where(
        df['session_seq'] > 0,
        df['rider_code'] + '_S' + df['session_seq'].astype(int).astype(str).str.zfill(4),
        pd.NA
    )

    # Optional: rows before first True should stay null
    df.loc[df['session_seq'] == 0, 'session_id'] = pd.NA

    # =========================
    # 3. SESSION SUMMARY
    # =========================
    session_df = df[df["session_id"].notna()].copy()

    def get_first_true_ts(x):
        mask = session_df.loc[x.index, "is_stationed"] == True
        return x[mask].min() if mask.any() else pd.NaT

    def get_last_false_ts(x):
        mask = session_df.loc[x.index, "is_stationed"] == False
        return x[mask].max() if mask.any() else pd.NaT

    session_summary = (
        session_df
        .groupby(["rider_code", "session_seq", "session_id"], as_index=False)
        .agg(
            session_start=("timestamp", get_first_true_ts),
            session_last_false=("timestamp", get_last_false_ts),
            session_last_event=("timestamp", "max"),
            total_logs=("id", "count"),
            total_true_logs=("is_stationed", lambda x: int((x == True).sum())),
            total_false_logs=("is_stationed", lambda x: int((x == False).sum()))
        )
    )

    # End of session:
    # if there is False in the session -> use last False
    # else -> fallback to last event in session
    session_summary["session_end"] = session_summary["session_last_false"].fillna(
        session_summary["session_last_event"]
    )

    session_summary["session_duration_minutes"] = (
        (session_summary["session_end"] - session_summary["session_start"]).dt.total_seconds() / 60
    ).round(2)

    return df, session_summary
    

# MAIN

## LOAD DATA

In [ ]:
detail_df, summary_df = create_mangkal_session()

summary_df['session_start'] = pd.to_datetime(summary_df['session_start'], unit='s')
summary_df['operating_date'] = summary_df['session_start'].dt.strftime('%Y-%m-%d')

In [ ]:
print("=== Detail Output ===")
detail_df.head(3)

In [ ]:
print("\n=== Session Summary ===")
summary_df.head(3)

In [ ]:
df_rider_total_mangkal_time = summary_df.groupby(['operating_date', 'rider_code']).agg(
    total_mangkal_time=('session_duration_minutes', 'sum'),
    total_mangkal_session=('session_id', 'nunique'),
).reset_index()

df_rider_total_mangkal_time

In [ ]:
df_rider_total_working_time = get_rider_total_working_time()

df_rider_total_working_time.head(3)

In [ ]:
df_raw_transactions = get_raw_transactions()

df_raw_transactions.head(3)

In [ ]:
df_hub_mapping = get_hub_mapping()

df_hub_mapping.head(3)

## TRANSFORM DATAFRAME

### Rider Daily Transactions

In [ ]:
df_daily_transactions = df_raw_transactions.groupby(['transaction_date', 'rider_code', 'rider_name', 'hub_code']).agg(
    total_sales=('total_price_after_discount', 'sum'),
    total_order=('id', 'nunique'),
    total_qty=('total_item_qty', 'sum')
).reset_index()

df_daily_transactions.head(3)

### Merge with Work & Mangkal Time

In [ ]:
# Normalize the operating date data type
df_daily_transactions_m = df_daily_transactions.rename(columns={'transaction_date': 'operating_date'}).copy()
df_rider_total_working_time_m = df_rider_total_working_time[['operating_date', 'rider_code', 'total_working_minutes']].copy()
df_rider_total_mangkal_time_m = df_rider_total_mangkal_time.copy()

for _df in (df_daily_transactions_m, df_rider_total_working_time_m, df_rider_total_mangkal_time_m):
    _df['operating_date'] = pd.to_datetime(_df['operating_date'], errors='coerce').dt.strftime('%Y-%m-%d')
    _df['rider_code'] = _df['rider_code'].astype(str).str.strip()


# Merge Final Data
df_final_summary = (
    df_daily_transactions_m.merge(
        df_rider_total_working_time_m,
        on=['operating_date', 'rider_code'],
        how='left'
    ).merge(
        df_rider_total_mangkal_time_m,
        on=['operating_date', 'rider_code'],
        how='left'
    )
    .merge(df_hub_mapping, on='hub_code', how='left')
)

numeric_cols = ['total_working_minutes', 'total_mangkal_time', 'total_mangkal_session', 'total_sales', 'total_order', 'total_qty']
for col in numeric_cols:
    df_final_summary[col] = pd.to_numeric(df_final_summary[col], errors='coerce').fillna(0)

string_cols = ['jilid', 'hub_name', 'business_unit', 'region', 'kota', 'provinsi']
for col in string_cols:
    if col in df_final_summary.columns:
        df_final_summary[col] = df_final_summary[col].fillna(0)

df_final_summary['mangkal_mins_pct'] = (
    df_final_summary['total_mangkal_time']
    / df_final_summary['total_working_minutes'].replace(0, np.nan)
    * 100
).round(2).fillna(0)

final_columns = [
    'operating_date',
    'rider_code',
    'jilid',
    'hub_name',
    'business_unit',
    'region',
    'kota',
    'provinsi',
    'total_working_minutes',
    'total_mangkal_time',
    'total_mangkal_session',
    'mangkal_mins_pct',
    'total_sales',
    'total_order',
    'total_qty',
]
df_final_summary = df_final_summary[final_columns]

df_final_summary.head(3)

In [ ]:
df_final_summary['batch_label'] = batch_label
df_final_summary['load_data_by'] = load_data_by

df_final_summary.head(3)

# LOAD TO DWH

## Engine

In [ ]:
from sqlalchemy import create_engine

engine = create_engine(
    "mysql+pymysql://{user}:{pw}@{host}/{db}?charset=utf8mb4".format(
        host=config_dl_dwh.hostname,
        user=config_dl_dwh.username,
        pw=config_dl_dwh.password,
        db=config_dl_dwh.db_target
    ),
    pool_pre_ping=True,
    pool_recycle=3600
)

## Insert to DB

In [ ]:
df_final_summary.to_sql(
    name="daily_rider_mangkal_session_summary",
    con=engine,
    if_exists="append",
    index=False,
    method="multi"
)
